# Chat Completion API로 프롬프트 엔지니어링 실습

Chat Completion API는 system, user, assistant 역할을 가진 메시지 목록을 모델에 전달하고 생성 응답을 받는 인터페이스이다. system 메시지는 모델의 역할과 공통 규칙을, user 메시지는 현재 요청과 입력 데이터를, assistant 메시지는 대화 이력을 표현한다. API 호출은 프롬프트를 코드로 재사용하고 결과 형식을 자동 처리할 수 있게 해 준다.

프롬프트 엔지니어링은 단순히 질문을 길게 쓰는 작업이 아니다. 역할, 입력 경계, 처리 규칙, 출력 스키마를 분리해 모델이 따라야 할 계약을 명확히 하는 작업이다. 강점은 빠른 실험과 다양한 업무 적용이며, 한계는 모델 출력이 확률적이고 외부 API 비용·속도·보안 제약이 있다는 점이다. API 키는 코드나 노트북에 저장하지 않고 실행 환경의 비밀 저장소에서 읽어야 한다.

이번 실습은 API 클라이언트를 준비한 뒤 기사 제목 교정, 상담형 응답, 레시피 제안, JSON 면접 질문 생성에 같은 메시지 구조가 어떻게 재사용되는지 확인한다. 외부 API를 호출하는 셀은 키·네트워크·모델 권한이 준비된 환경에서만 실행하고, 응답 내용은 업무 규칙과 JSON 파싱 결과로 다시 검증한다.


### OpenAI Python SDK 설치

이 셀은 Chat Completion API를 호출하기 위한 `openai` 패키지를 설치한다. 설치가 끝나면 이후 셀의 `OpenAI` 클래스를 가져올 수 있다. 패키지 버전과 인터넷 연결이 필요한 환경 준비 단계이므로, 이미 설치된 수업 환경에서는 실행 결과가 달라질 수 있다.


In [1]:
# %pip install -U openai python-dotenv

## PyCharm 환경 설정

1. PyCharm에서 `08_llm` 폴더를 프로젝트로 연다.
2. PyCharm Terminal에서 `python -m pip install openai python-dotenv`를 실행한다.
3. `08_llm/.env` 파일에 `OPENAI_API_KEY`를 저장한다.
4. 키 값은 코드셀, 출력, Git 또는 공유 파일에 포함하지 않는다.


### PyCharm 프로젝트의 `.env` 설정 불러오기

`.env`는 API 키와 실행 설정을 노트북 코드에서 분리하는 로컬 파일이다. `find_dotenv(usecwd=True)`는 현재 Jupyter 작업 폴더부터 상위 폴더로 이동하며 `08_llm/.env`를 찾고, `load_dotenv()`는 그 값을 현재 커널의 환경 변수로 불러온다.

이 셀은 `OPENAI_API_KEY` 변수가 준비되었는지만 검사하고 실제 값은 출력하지 않는다. 이후 OpenAI SDK와 연동 라이브러리는 환경 변수를 자동으로 사용한다.


In [2]:

import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError(
        "08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 현재 셀을 다시 실행하세요."
    )

load_dotenv(dotenv_path, override=False)

required_env_vars = [
    "OPENAI_API_KEY",
]

missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    
    missing_names = ", ".join(missing_env_vars)
    raise RuntimeError(f".env 파일의 다음 변수를 확인하세요: {missing_names}")

print("환경 변수 준비 완료")

환경 변수 준비 완료


### API 클라이언트 초기화

`OpenAI` 객체는 이후 모든 API 요청의 진입점이다. 앞 셀에서 읽은 키를 `api_key`에 전달해 `client`에 저장하고, 뒤의 프롬프트 함수들이 이 클라이언트를 재사용한다. 객체 생성은 네트워크 요청 자체가 아니지만, 실제 호출은 유효한 키와 사용 권한을 요구한다.


In [3]:
from openai import OpenAI

client = OpenAI()

### 가장 작은 Chat Completion 요청 만들기

이 셀은 `messages` 목록 안에 system과 user 메시지를 넣어 모델에 한 번 요청한다. `model`은 사용할 모델 이름, `temperature`는 생성 다양성, `max_completion_tokens`는 생성 길이 상한을 뜻한다. 응답 객체는 다음 셀에서 `choices[0].message.content`로 꺼낸다. 외부 API 호출이므로 실행하면 인증·네트워크·모델 사용 가능 여부를 함께 확인한다.


In [4]:
# OpenAI의 Chat Completions API에 대화를 전달해서 챗봇 답변을 생성하는 Python 코드입니다.

response = client.chat.completions.create(
    model="gpt-4.1-mini",

    # 전달할 대화
    messages=[
        {
            "role": "system",  # 모델이 대화 전체에서 따를 역할과 규칙
            "content": [
                {
                    "type": "text",
                    "text": "너는 아주 친절하고, 많은 도움을 주는 챗봇이야",
                }
            ],
        },
        {
            "role": "user",  # 사용자 요청
            "content": [
                {
                    "type": "text",
                    "text": "안녕~ 내 이름은 김나은이야.",
                }
            ],
        },
    ],

    # 응답 형식 지정
    response_format={"type": "text"},
    temperature=1,                # 값이 높을수록 무작위성 증가
    max_completion_tokens=2048,   # 텍스트 생성 상한
    top_p=1,                      # 누적 확률 범위 제한 없음
    frequency_penalty=0,          # 같은 표현 반복에 대한 불이익
    presence_penalty=0,           # 이미 등장한 주제에 대한 불이익
)

### 응답 객체에서 생성 텍스트 추출하기

응답의 `choices`는 생성 후보 목록이며, 이 예제는 첫 번째 후보의 `message.content`를 표시한다. 후보가 비어 있거나 API 호출이 실패하면 이 접근은 불가능하므로, 앞 셀의 정상 응답 여부를 먼저 확인해야 한다. 출력 텍스트는 모델의 생성물이지 검증된 사실이 아니므로 업무 규칙과 원문을 대조한다.


In [5]:
print(response.choices[0].message.content)

안녕, 김나은! 만나서 반가워. 오늘 기분은 어때? 도움이 필요하면 언제든 말해줘!


## 프롬프팅의 기본구성

https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/

1. Instruction 지시사항
2. Context 문맥
3. Input Data/Example 입력/예시
4. Output Indicator 출력지시

## 기사 제목 교정

- 기자들이 송고한 기사에서 제목을 추출하고, 표현 조정
- 프랑스AFP 속보시스템에서 도입되어 사용


### 기사 제목 교정 프롬프트 실행

이 셀은 역할과 교정 규칙을 `system_message`에, 실제 기사 제목을 `user_message`에 분리해 전달한다. 예시와 출력 형식은 모델이 두 제목을 정해진 구조로 반환하도록 돕는다. `print` 결과에서 비속어 완화, 핵심 정보 보존, 지정된 두 줄 형식이 모두 지켜졌는지 확인한다. API 응답은 실행 환경에 따라 달라지며 결과를 실제 관찰한 값처럼 미리 단정하지 않는다.


In [6]:
# 교정이 필요한 제목
title_before = '테이의 FM 개꿀 라디오 방송에 주목해주세요.'
# title_before = '졸라 빡센 작업으로 끼니 거르기 일쑤인 노동자들의 애환'

# system 메시지에는 모든 제목에 공통으로 적용할 역할, 절차, 출력 형식, 예시를 넣는다.
system_message = """
기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

- 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
- 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
- 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
- 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

### Steps ###
1. 기사제목을 읽고 주요내용을 이해하세요.
2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


### Output Format ###
기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

- 원래 제목: [기사 원래 제목]
- 교정 제목: [교정 기사 제목]

### Examples ###
- 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
- 교정 제목: "서울 대형 화재, 수백명 대피"

- 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
- 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

### Extra Instructions ###
- 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
- 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
- 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

"""

# 이번 요청에서만 바뀌는 기사 제목을 user 메시지에 삽입
user_message = f"""
다음 기사 제먹을 교정해주세요.

제목 : {title_before}
"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",  # 제목 교정에 사용할 모델 ID이다.
    messages=[
        {
            "role": "system",  # 위에서 만든 공통 교정 규칙을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ]
        },
        {
            "role": "user",  # 교정할 제목이 포함된 현재 요청을 전달한다.
            "content": [
                {
                    "type": "text",
                    "text": user_message
                }
            ]
        }
    ],
    response_format={
        "type": "text"  # 결과를 일반 문자열로 받는다.
    },
    temperature=1,  # 표현의 다양성을 조절한다.
    max_completion_tokens=2048,  # 교정 결과가 사용할 수 있는 생성 토큰 상한이다.
    top_p=1,  # 후보 토큰의 누적 확률 범위를 제한하지 않는다.
    frequency_penalty=0,  # 동일 표현 반복에 대한 추가 패널티를 사용하지 않는다.
    presence_penalty=0  # 새로운 주제 사용을 강제로 유도하지 않는다.
)

# 응답 결과 확인
print(response.choices[0].message.content)

- 원래 제목: 테이의 FM 개꿀 라디오 방송에 주목해주세요.
- 교정 제목: 테이의 FM 라디오 방송, 놓치지 마세요!


### 반복 요청을 제목 교정 함수로 리팩터링하기

`correct_news_title`은 제목, 모델 이름, `temperature`, `top_p`를 인자로 받아 같은 프롬프트 구조를 재사용한다. 함수의 반환값은 응답 텍스트이며, 호출한 쪽은 `output`에 저장해 출력하거나 후속 검증에 사용한다. 같은 제목을 여러 설정으로 비교할 때는 입력을 고정하고 생성 파라미터만 바꿔 형식 준수와 표현 차이를 비교한다.


In [7]:
def correct_news_title(title_before, model='gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
    기사들이 송고한 제목에서 맞춤법, 문법, 의미, 어조등에 있어서 교정작업을 수행해 주세요.

    - 기사 제목이 명확하고 주제와 잘 맞도록 조정하세요.
    - 독자의 관심을 끌 수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.
    - 어조가 지나치게 감정적이거나 부정적인 경우 표현을 완화하거나, 중립적인 어조로 수정하세요.
    - 비속어가 포함되어 있는 경우, 비속어를 반드시 제거하고, 의미를 적절히 유지하도록 제목을 교정하세요.

    ### Steps ###
    1. 기사제목을 읽고 주요내용을 이해하세요.
    2. 제목이 전달하고자하는 메세지를 명확하게 반영하는지 검토하세요.
    3. 맞춤법, 문법, 의미 전달의 정확성등을 점검하고 적절히 수정하세요.
    4. 제목이 자연스럽고, 독자에게 매력적으로 다가갈수 있는지 점검하고 매우 간결하게 정리하세요.


    ### Output Format ###
    기사 원래 제목과 교정된 제목을 다음 형식으로 제공하세요.

    - 원래 제목: [기사 원래 제목]
    - 교정 제목: [교정 기사 제목]

    ### Examples ###
    - 원래 제목: "어제 서울에서 큰 불이 나 수백명이 대피했다."
    - 교정 제목: "서울 대형 화재, 수백명 대피"

    - 원래 제목: "전기자동차 판매량 급감에 내연차회사들이 즐거워하는 중입니다."
    - 교정 제목: "전기차 판매량 급감에 웃는 내연차회사들"

    ### Extra Instructions ###
    - 제목이 너무 길면, 간결하게 줄이되, 핵심 메세지를 잃어버려서는 안됩니다.
    - 지역명, 시간 등의 중요한 정보는 명확하게 유지하세요.
    - 제목이 특정집단이나 대상에 대해 중립적이지 않을 경우, 그 표현을 완화하세요.

    """
    user_message = f"""
    다음 기사제목을 교정해주세요.

    제목: {title_before}
    """

    response = client.chat.completions.create(
        model=model,  # 호출자가 선택한 모델 ID이다.
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": system_message
                    }
                ]
            },
            {
                "role": "user",  # 이번에 교정할 제목이다.
                "content": [
                    {
                    "type": "text",
                    "text": user_message
                    }
                ]
            }],
            response_format={
                "type": "text"
            },
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )
    # 호출부가 SDK 응답 구조를 몰라도 되도록 생성된 문자열만 반환한다.
    return response.choices[0].message.content

title_before = '폭염 에바띠~ 사람이 살 수 있는 날씨 맞냐긔;;;'
output = correct_news_title(title_before)
print(output)

- 원래 제목: 폭염 에바띠~ 사람이 살 수 있는 날씨 맞냐긔;;;
- 교정 제목: 폭염, 사람이 살기 힘든 날씨 지속


## 연애코치 ReAct

### ReAct 형식의 상담 프롬프트 함수 정의

ReAct는 추론과 행동을 명시적으로 구분해 복잡한 작업을 단계적으로 수행하게 하는 프롬프팅 방식이다. 이 예제는 외부 도구 호출을 구현하지 않고 상황 분석·행동 계획·실행이라는 출력 구조를 요청한다. `dating_coach`는 사용자 고민 문자열을 받아 응답 텍스트를 돌려주며, 다음 두 셀이 서로 다른 입력에서 구조가 유지되는지 확인한다.


In [12]:
def dating_coach(prompt, model='gpt-4o-mini', temperature=1, top_p=1):
    # System message: 지침 설정
    system_message = """
    << System Instruction >>
    어떤 상황에서든 최고의 논리적/감성적 관점을 적용하는 연애코치로써 사용자의 고민을 해결해 주세요.

    << Output Format >>
    1. 상황분석:

    2. 행동계획:

    3. 실행:
    """

    # User message: 사용자 입력 프롬프트를 일정 형식으로 가공한 형태
    user_message = f"""
    사용자 현재 현황:
    {prompt}
    """

    # Chat Completion 요청
    response = client.chat.completions.create(
        model = model,
        messages = [
            {
                "role" : "system",
                "content" : [{"type":"text", "text": system_message}]
             },
            {
                "role" : "user",
                "content" : [{"type":"text", "text": user_message}]
             }
        ],
        response_format = {"type":"text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    # 첫 번째 후보의 상담 결과 텍스트만 반환
    return response.choices[0].message.content

### 첫 번째 상담 입력으로 출력 형식 확인

정의한 `dating_coach`에 기념일 선물이라는 한 문장 입력을 전달한다. 출력에서 상황 분석, 행동 계획, 실행 항목이 구분되는지와 안전하지 않거나 과도하게 단정적인 조언이 없는지를 확인한다. 실제 결과는 모델과 실행 시점에 따라 달라진다.


In [13]:
prompt = """
저 이번 주말에 여자친구와 100일인데, 기억에 남는 선물을 하고 싶다. 뭐가 좋을까?
"""

print(dating_coach(prompt))

1. 상황분석:  
현재 여자친구와의 관계가 100일인 만큼, 특별하고 의미 있는 선물이 필요합니다. 상대방의 취향이나 기호를 고려하면서도, 두 사람의 특별한 순간을 기념할 수 있는 방법이 중요합니다. 그녀와의 추억이나 관심사를 반영한 선물이 효과적일 것입니다.

2. 행동계획:  
- 그녀의 취향 확인: 그녀가 좋아하는 색상, 스타일, 혹은 관심사를 파악하세요.
- 특별한 순간 회상: 처음 만났던 날 또는 특별한 순간들을 상기시켜 그와 관련된 선물을 고려하세요.
- 직접적인 경험: 단순한 물건보다는 두 사람이 함께할 수 있는 경험을 고려해보세요.

3. 실행:  
몇 가지 선물 아이디어를 다음과 같이 제안합니다:
- **커플 액세서리**: 두 사람의 이름이나 기념일이 새겨진 커플 목걸이나 팔찌.
- **특별한 데이트**: 그녀가 좋아하는 레스토랑에서의 저녁 식사 예약 또는 아름다운 장소에서의 피크닉.
- **기억의 앨범**: 두 사람의 사진 및 추억을 담은 스크랩북이나 포토 북을 만들어 보는 것도 좋습니다.
- **손편지 작성**: 진솔한 마음을 담아 쓴 편지는 언제나 특별함을 더해줍니다.

이런 선물과 함께 특별한 하루를 계획해 보세요. 사랑은 경험과 기억 속에서 더욱 깊어질 수 있습니다.


### 두 번째 상담 입력으로 일반화 범위 확인

같은 함수를 다른 갈등 상황에 적용해 프롬프트 구조가 입력 변화에도 유지되는지 확인한다. 두 응답을 비교할 때는 문장 길이보다 상황에 맞는 행동 제안과 출력 형식 준수 여부를 기준으로 삼는다. 개인 관계 조언은 사실 판단이나 전문 상담을 대체하지 않는다는 한계도 함께 안내한다.


In [14]:
prompt = """
내가 친구들과 놀다가 술에 취해 길거리에 잠들어 버렸어. 이 사실을 안 여자친구는 단단히 화가 난 상태야.
어떻게 여자친구의 기분을 풀어주는 게 좋을까?
"""

print(dating_coach(prompt))

1. 상황분석:
   - 당신은 친구들과의 술자리에서 과음하여 길거리에서 잠드는 상황이 발생했습니다. 
   - 여자친구는 당신이 안전을 위협 받고 있을 뿐 아니라, 그로 인해 그녀에게 걱정과 불안감을 주었기 때문에 화가 난 상태입니다.

2. 행동계획:
   - 진심어린 사과: 먼저 어떤 이유에서 그런 일이 일어났는지 설명하고, 그녀의 걱정과 불안을 충분히 이해하고 있다는 것을 표현하세요.
   - 감정 공유: 그녀의 불편한 감정을 인정하고, 그에 대해 공감하는 대화를 나누세요.
   - 책임감 있는 태도 보여주기: 앞으로는 안전을 더 우선시하겠다는 다짐과 구체적인 행동 계획을 세워서 이야기를 하세요.
   - 추가적인 배려: 그녀를 위해 작은 선물이나 메모를 주는 것도 좋은 방법이 될 수 있습니다.

3. 실행:
   - 먼저, 개인적으로 그녀에게 연락하여 사과의 말을 전합니다. “내 행동 때문에 걱정을 끼쳐서 정말 미안해. 너의 기분이 얼마나 상했을지 이해해. 앞으로 더 조심할게”라고 말합니다.
   - 만나서 그녀에게 진심으로 이야기를 나누고, 그녀의 이야기에 귀 기울이며 진정한 감정을 교환합니다.
   - 이후, 두 사람이 즐길 수 있는 작은 데이트를 계획하거나 그녀가 좋아하는 음식 또는 간단한 선물을 준비하여 기분을 전환시켜 줄 기회를 만듭니다. 
   - 마지막으로, 앞으로의 약속을 통해 신뢰를 회복하고, 반복되지 않도록 노력할 것을 다짐하십시오.


## 냉털마스터 ReAct
- 사용자는 냉장고에 남아있는 음식재료를 알려주면, LLM은 이를 바탕으로 어떤 음식을 만들지를 조언해준다.
- Reasoning/Action을 끌어낼수 있는 적절한 프롬프팅을 작성한다.


### 재료 기반 레시피 제안 함수 정의

이 함수는 재료 목록을 user 메시지에 넣고, 분석·계획·검증·최종 레시피 순서의 응답을 요청한다. 리스트인 `user_foods`는 f-string에서 문자열로 변환되어 프롬프트의 입력 맥락이 된다. 모델은 실제 식재료 상태나 알레르기를 알 수 없으므로, 생성한 레시피는 조리 전 안전성·보관 상태·알레르기 정보를 사용자가 다시 확인해야 한다.


In [15]:
def fridge_raid_master(user_foods, model = 'gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
당신은 사용자 냉장고의 재료를 가지고 최고의 음식을 만들 수 있는 레시피를 추천하는 챗봇입니다.
현재 상황을 분석하고 실행계획을 세우며 추가 내용이 있을지 확인하는 꼼꼼함을 보여주세요.

## 지시사항

### 상황분석
1. **재료 확인**: 냉장고에 있는 재료 목록을 작성하고, 각 재료의 신선함과 사용 가능 시간을 파악합니다.
2. **요리 컨셉 설정**: 재료를 고려하여 요리의 주제를 정하고 잠재적인 요리 아이디어를 계획합니다.

### 실행계획
3. **목표 변경**: 사용 가능한 재료로 만들고자 하는 요리의 목표를 설정하고, 필요한 정보나 추가 자료를 조사합니다.
4. **전략 개발**: 단계별로 목표를 달성하기 위한 전반적인 요리 계획을 세웁니다.
    단, 사용자가 따라하기 쉽게 단계별 가이드를 작성해야 합니다.

### 검증 및 추가내용 확인
5. **레시피 검토**: 작성된 레시피를 검토하여 정보가 명확하고 완전한지 확인합니다.
6. **수정**: 필요 시 수정하여 최종 레시피를 완성합니다.

## 출력형식

1. 상황분석:
   - 재료 목록 및 상태
   - 요리 컨셉 및 아이디어

2. 실행계획:
   - 목표 및 세부 계획
   - 필요한 추가 자료 및 전략

3. 검증 및 추가내용 확인
   - 레시피의 정확성 검토
   - 수정 및 편집

4. 최종레시피
   - 사용자가 따라하기 쉽게 목록으로 작성
   - 필요한 준비물로 별도록 작성할것!

## Examples

- **Input**:
    사용자의 냉장고에는 현재 [무, 파, 두부]이/가 있습니다.
-   **Output**:
  1. 상황분석: 오늘은 냉장고에 있는 무, 파, 두부를 이용해 한식 스프를 만들어보겠습니다...
  2. 실행계획: 먼저 무와 파를 얇게 썰어 냄비에 넣고 물을 부어...
  3. 검증 및 추가내용 확인: 레시피를 확인한 결과, 무와 두부의 사용 방법에 대한 설명이 추가로 필요합니다...

## Notes

- 요리의 문화적 관련성과 감수성을 고려합니다.
- 레시피는 예상 독자의 요리 지식 수준에 맞게 조정됩니다.

    """

    user_message = f"""
    사용자의 냉장고에는 현재 {user_foods}이/가 있습니다.
    """

    # Chat Completion 요청
    response = client.chat.completions.create(
        model = model,
        messages = [
            {
                "role" : "system",
                "content" : [{"type":"text", "text": system_message}]
             },
            {
                "role" : "user",
                "content" : [{"type":"text", "text": user_message}]
             }
        ],
        response_format = {"type":"text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### 재료 목록을 Markdown 응답으로 표시하기

`Markdown`과 `display`는 모델이 반환한 마크다운 형식의 레시피를 노트북에서 읽기 쉽게 렌더링한다. `user_foods`가 함수 입력이고 반환 문자열이 `Markdown`의 입력이 된다. 화면에서는 네 개의 출력 구역이 실제로 구분되는지와 재료 목록이 모두 반영되었는지 확인한다.


In [17]:
from IPython.display import Markdown

user_foods = ['먹다 남은 치킨', '스모크 햄', '토마토 소스', '양파', '목이버섯', '낫또', '배추', '바나나']
display(Markdown(fridge_raid_master(user_foods)))

1. **상황분석**:
   - **재료 목록 및 상태**:
     - **치킨**: 먹다 남은 상태이므로 가급적 빨리 사용해야 함.
     - **스모크 햄**: 보관 기간이 길며 맛이 좋은 상태.
     - **토마토 소스**: 개봉 후 어느 정도 사용이 가능, 확인 후 사용.
     - **양파**: 신선하고 다양한 요리에 활용 가능.
     - **목이버섯**: 수분감이 있을 수 있으니 신선한 상태 확인 필요.
     - **낫또**: 일반적으로 장기간 보관 가능, 냉장고에서 곰팡이가 안 생기면 사용 가능.
     - **배추**: 상한 부분 없이 신선하다면 사용 가능.
     - **바나나**: 상태에 따라 갈변이 있을 수 있으니 확인 필요.

   - **요리 컨셉 및 아이디어**:
     오늘은 다양한 치킨과 햄을 활용해 '치킨과 햄 파스타 또는 볶음밥'을 만드는 것을 제안합니다. 토마토 소스를 이용해 감칠맛을 더하고, 양파와 목이버섯은 식감과 영양을 보강합니다. 낫또는 별도의 반찬 또는 올려서 즐길 수 있는 선택이 될 수 있습니다.

2. **실행계획**:
   - **목표 및 세부 계획**:
     - 치킨과 스모크 햄을 활용하여 간단하고 맛있는 볶음밥을 만들 것.
     - 토마토 소스를 사용해 볶음밥에 풍미를 추가.
     - 양파와 목이버섯을 함께 볶아 다채로운 식감을 연출.
   
   - **필요한 추가 자료 및 전략**:
     - 요리법 베이스: 볶음밥 또는 파스타 만드는 방법.
     - 양념: 간장, 소금, 후춧가루, 올리브유 또는 식용유 등 제공할 재료 확인.
     - 조리 시간: 20-30분 내외로 설정.

3. **검증 및 추가내용 확인**:
   - 볶음밥 레시피에서 각 재료가 어떻게 조화를 이루는지 명확하게 설명.
   - 목이버섯은 미리 불려서 사용할 것인지 여부를 추가.
   - 필요시 양념 비율을 다양하게 제안.

4. **최종레시피**:
   - **준비물**:
     - 먹다 남은 치킨 1컵
     - 스모크 햄 1컵
     - 양파 1개
     - 목이버섯 1/2컵
     - 토마토 소스 1컵
     - 기존 밥 2컵 또는 파스타
     - 소금, 후춧가루, 식용유(또는 올리브유)

   - **조리 과정**:
     1. 양파를 잘게 썰고, 목이버섯은 미리 불려서 물기를 제거합니다.
     2. 팬에 식용유를 두르고 양파를 볶아 향이 올라오면 스모크 햄을 추가합니다.
     3. 다진 치킨을 팬에 넣고 잘 섞어 볶습니다.
     4. 목이버섯을 넣고 잘 섞어가며 볶습니다.
     5. 밥(또는 삶은 파스타)을 넣고 가볍게 볶아줍니다.
     6. 토마토 소스를 넣고 모든 재료가 잘 섞일 때까지 볶습니다.
     7. 소금과 후춧가루로 간을 맞추고, 취향에 따라 낫또를 올려 서빙합니다.

이제 레시피를 진행하여 맛있는 요리를 즐겨보세요! 추가적인 질문이 있다면 언제든지 알려주세요.

## 면접질문 생성 JSON 출력


### JSON 형식 면접 질문 생성 함수 정의

이 함수는 채용공고 문자열을 받아 hard skill과 soft skill 질문·답변을 담은 JSON 문자열을 요청한다. `response_format={"type": "json_object"}`는 JSON 객체 형식의 응답을 요구하지만, 실제 파싱 전에는 필수 키와 값의 타입을 검증해야 한다. 함수는 아직 API를 호출하지 않으며 다음 셀의 채용공고가 입력으로 전달될 때 호출된다.


In [18]:
def job_interview(job_posting, model='gpt-4o-mini', temperature=1, top_p=1):
    system_message = """
당신은 머신러닝/딥러닝/LLM/AI서비스개발의 전문가로써, 해당분야의 으뜸가는 면접관입니다.
매번 그룹사의 인재를 발굴하기 위해 면접질문/모범답안을 작성하고 있습니다.

<<지시사항>>
- 사용자가 제출한 채용공고의 내용을 바탕으로 예상면접질문과 답변을 작성해주세요.
- 하드스킬과 소프트스킬 두개의 섹션으로 나누어 작성해주세요.
- 각 스킬별로 질문/답변을 3개씩 만들어주세요.

<<출력형식>>
출력은 json형식으로만 반환되어야 합니다.

{{
    "hard_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ],
    "soft_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ]
}}
"""
    user_message = f"""
    채용 공고 : {job_posting}
"""

    # Chat Completion 요청
    response = client.chat.completions.create(
        model = model,
        messages = [
            {
                "role" : "system",
                "content" : [{"type":"text", "text": system_message}]
             },
            {
                "role" : "user",
                "content" : [{"type":"text", "text": user_message}]
             }
        ],
        response_format = {"type":"text"}, # 응답 타입 지정
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### 채용공고를 입력으로 전달하고 원본 응답 확인

긴 `job_posting` 문자열은 모델이 근거로 삼을 입력 데이터이며, `output`은 함수가 반환한 JSON 문자열이다. 먼저 `type(output)`이 `str`인지와 내용이 JSON 객체처럼 보이는지 확인한 뒤 다음 셀에서 파싱한다. 실제 API 호출은 비용과 네트워크가 발생하므로 수업 환경의 키와 사용 한도를 확인한 뒤 실행한다.


In [ ]:
job_posting = """
| 모집부문                      | 담당업무                                                                                                                                                                                                                                                                                               | 자격요건 및 필수사항                                                                                                                                                                                                                                                                                                                                                                 |
| ------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **AI 인프라 구축 (ML/DEVOps)** | 1. AI 모델의 배포를 위한 인프라 구축 및 운영2. 알고리즘 개발자에 대한 기술 지원3. 머신러닝 인프라의 설계 및 개발, 운영4. 모니터링, 로깅 시스템의 개발 및 구현5. 머신러닝 시스템의 학습, 배포, 테스트 환경 구축, 운영 및 자동화6. 오픈소스 기반의 플랫폼 통합 및 내재화 구축(MLflow, LakeFS, CVAT, BentoML 등)7. 보안, 인증, 권한 관리 등 운영 정책 및 관리자 기능 구축8. 시스템과 프로세스에 대한 문서 작업 |
**학력**- 대졸이상(4년제)**경력**- 신입**필수사항**- Python 프로그래밍 능력- 컨테이너 오케스트레이션 툴 활용 능력(Docker, Kubernetes 등)- Jenkins, ArgoCD, Gitlab 등 구축 및 운영 경험**우대사항**- DevOps 환경 구축 혹은 DevOps 환경에서 개발 및 배포 경험 보유자- MLflow, Kubeflow, ML 모델 서빙 플랫폼 등과 관련된 실무 경험 보유자- Kubernetes, Docker, GitOps, CI/CD, Vault, MinIO 등 클라우드 네이티브 인프라 경험 보유자 |
"""

print(job_interview(job_posting))

### 지시문과 입력 데이터를 분리한 JSON 프롬프트

이 셀은 시스템 역할 설명과 사용자 쪽의 지시·채용공고를 분리해 같은 JSON 생성 함수를 다시 정의한다. 프롬프트 구성 요소를 분리하면 역할 규칙은 재사용하고 채용공고만 바꿀 수 있다. 두 함수 정의는 같은 이름을 사용하므로 이 셀을 실행하면 앞 셀의 정의를 덮어쓴다는 점을 확인한다.


In [20]:
def job_interview(job_posting, model='gpt-4o-mini', temperature=1, top_p=1):
    # system 메시지에는 여러 요청에서 유지할 면접관 역할만 둔다.
    system_message = """
당신은 머신러닝/딥러닝/LLM/AI서비스개발의 전문가로써, 해당분야의 으뜸가는 면접관입니다.
매번 그룹사의 인재를 발굴하기 위해 면접질문/모범답안을 작성하고 있습니다.
"""

    # 작업 지시, JSON 예시, 실제 채용공고를 한 user 메시지의 경계 표식으로 구분한다.
    # {{와 }}는 f-string이 JSON 중괄호를 변수 자리로 해석하지 않게 한다.
    user_message = f"""

<<지시사항>>
- 사용자가 제출한 채용공고의 내용을 바탕으로 예상면접질문과 답변을 작성해주세요.
- 하드스킬과 소프트스킬 두개의 섹션으로 나누어 작성해주세요.
- 각 스킬별로 질문/답변을 3개씩 만들어주세요.

<<출력형식>>
출력은 json형식으로만 반환되어야 합니다.

{{
    "hard_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ],
    "soft_skill": [
        {{
            "question": "",
            "answer": "",
        }}
    ]
}}

<< 채용공고 >>
{job_posting}
"""
    # 역할 메시지와 작업 메시지를 결합하고 JSON mode로 응답을 요청한다.
    response = client.chat.completions.create(
        model=model,  # 함수 호출 시 선택한 모델 ID이다.
        messages=[
            {
                "role": "system",
                "content": [
                    {
                    "type": "text",
                    "text": system_message
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                    "type": "text",
                    "text": user_message
                    }
                ]
            }],
            # JSON 문법은 보장하지만 hard_skill 같은 필드 존재 여부는 코드에서 별도로 검사해야 한다.
            response_format={
                "type": "json_object"
            },
        temperature=temperature,
        max_completion_tokens=2048,
        top_p=top_p,
        frequency_penalty=0,
        presence_penalty=0
    )

    return response.choices[0].message.content

### JSON 문자열을 Python 자료구조로 변환하기

`json.loads`는 JSON 문자열 `output`을 Python 딕셔너리로 변환한다. 변환 뒤 `hard_skill`과 `soft_skill` 키로 질문·답변 목록을 꺼내 후속 화면 표시, 파일 저장, 평가에 사용할 수 있다. JSON 문법이 깨지거나 필수 키가 없으면 오류가 나므로, API 응답 형식과 키 존재를 검증하는 것이 실무에서 중요하다.


In [24]:
import json # json -> dict 변환

output = job_interview(job_posting)
data = json.loads(output)

print(type(data)) # 타입이 dict 변환 되었느지 확인

hard_skill_qa = data['hard_skill']
soft_skill_qa = data['soft_skill']

print(hard_skill_qa)
print('-' * 100)
print(soft_skill_qa)

<class 'dict'>
[{'question': 'Python 프로그래밍 언어를 사용하여 머신러닝 인프라를 구축할 시 가장 중요한 점은 무엇인가요?', 'answer': '가장 중요한 점은 코드의 가독성과 유지보수성입니다. 머신러닝 인프라는 복잡한 시스템이기 때문에, 다른 팀원이나 후속 개발자가 쉽게 이해하고 수정할 수 있도록 구조화된 코드를 작성하는 것이 중요합니다. 또한, 라이브러리와 프레임워크를 적절히 사용하여 개발 속도를 높이는 것이 필요합니다.'}, {'question': 'Docker와 Kubernetes를 이용한 컨테이너 오케스트레이션 경험에 대해 말씀해 주세요.', 'answer': 'Docker를 사용하여 애플리케이션을 컨테이너화하고, Kubernetes로 배포 및 스케일링을 관리한 경험이 있습니다. 예를 들어, 여러 마이크로서비스를 컨테이너로 패키징하고, Kubernetes 클러스터에서 서로 다른 서비스들 간의 통신을 설정하여 원활한 CI/CD 파이프라인을 만들었습니다.'}, {'question': 'CI/CD 도구(Jenkins, ArgoCD 등)를 통한 자동화 경험에 대해 설명해 주세요.', 'answer': 'Jenkins를 사용하여 빌드와 테스트 프로세스를 자동화한 경험이 있습니다. 아울러, ArgoCD를 활용한 GitOps 환경에서도 배포 파이프라인을 구성하여 코드 변경 시 자동으로 애플리케이션을 배포할 수 있는 시스템을 구축했습니다. 이로 인해 배포 시간이 크게 감소하였습니다.'}]
----------------------------------------------------------------------------------------------------
[{'question': 'AI 인프라 구축 과정에서 팀원들과의 협업은 어떻게 진행하셨나요?', 'answer': '상시 커뮤니케이션을 통해 진행 상황을 공유하고, 정기적인 회의를 통해 문제를 논의했습니다. 팀원 각자의 역할을 명확히 하여, 각자 맡은 부분에 대한 책임감을